# GEMAMX GROMACS - Piloto corto en Colab

Esta libreta arma un piloto corto y realista para validar el circuito completo `Vina -> Colab -> JSON -> app web`.

Pares incluidos:
- `PIK3CA_H1047R + Alpelisib`
- `PIK3CA_H1047R + Pictilisib`
- `ERBB2_V777L + Pictilisib`
- `ERBB2_V777L + Alpelisib`
- `TP53_R248Q + Abemaciclib`
- `TP53_R248Q + Pictilisib`


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
%cd /content/drive/MyDrive/GEMAMX


In [ ]:
# Instalacion sugerida de base para Colab Linux
!apt-get -qq update
!apt-get -qq install -y gromacs openbabel
!pip -q install pandas numpy matplotlib


In [ ]:
!python gromacs/build_gromacs_batch.py --top_n_per_variant 5
!python gromacs/build_gromacs_pilot_batch.py


In [ ]:
import json
from pathlib import Path

from gromacs.colab_pilot_utils import load_manifest, scaffold_pair

pilot_manifest = load_manifest('gromacs/results/gromacs_pilot_manifest.json')
pairs = pilot_manifest['pairs']
print(json.dumps(pilot_manifest['summary'], indent=2, ensure_ascii=False))
pairs


In [ ]:
BASE_ROOT = Path('/content/drive/MyDrive/GEMAMX')
scaffolded = [scaffold_pair(pair, BASE_ROOT, md_time_ns=20.0) for pair in pairs]
scaffolded


## Que deja esta fase

Para cada par se crea un directorio en `gromacs/runs/<tag>` con:
- archivos de entrada copiados
- `em.mdp`, `nvt.mdp`, `npt.mdp`, `md.mdp`
- `run_gromacs.sh`
- `pair_metadata.json`

El script ya contiene el flujo base de GROMACS, pero la topologia del ligando sigue siendo el punto a completar con ACPYPE o CGenFF.


In [ ]:
# Vista rapida del primer script generado
first_script = Path(scaffolded[0]['script_path'])
print(first_script)
print(first_script.read_text(encoding='utf-8')[:3000])


## Ejecucion manual por par

Una vez que resuelvas la topologia del ligando, puedes correr un par asi:

```bash
cd /content/drive/MyDrive/GEMAMX/gromacs/runs/PIK3CA_H1047R__Alpelisib
bash run_gromacs.sh
```


In [ ]:
# Placeholder inicial de resultados mientras se completan corridas reales
from datetime import datetime
from gromacs.colab_pilot_utils import aggregate_pair_metrics

results = []
for pair in pairs:
    results.append({
        'tag': pair['tag'],
        'variant_id': pair['variant_id'],
        'drug_name': pair['drug_name'],
        'vina_score_kcalmol': pair['vina_score_kcalmol'],
        'status': 'scaffolded',
        'run_dir': str(BASE_ROOT / pair['recommended_colab_run_dir']),
        'trajectory_path': None,
        'plot_paths': {},
        'notes': 'Scaffold creado en Colab. Falta completar topologia del ligando y correr GROMACS.'
    })

out = aggregate_pair_metrics(
    results,
    BASE_ROOT / 'gromacs/results/gromacs_metrics.json',
    parameters={
        'force_field': 'charmm36m',
        'water_model': 'tip3p',
        'md_time_ns': 20,
        'temperature_K': 300,
        'pilot_mode': True,
        'generated_at_colab': datetime.now().isoformat()
    },
    pipeline_version='GEMAMX-GROMACS-v2-pilot'
)
print(out)


## Regreso a local

Cuando ya tengas `gromacs/results/gromacs_metrics.json` actualizado en Drive, vuelve a Windows y corre:

```powershell
& 'C:\\Users\\30046\\AppData\\Local\\Programs\\Python\\Python312\\python.exe' D:\\ProyectosIA\\GEMAMX\\gromacs\\ingest_gromacs_results.py
```

Despues recarga `http://localhost:8000/` y abre `Dinamica Molecular`.
